# Extract parish ministries

Load the parish list generated by `01_build_parish_list.ipynb` as the starting point for ministry extraction. Run this notebook from the project root.


In [4]:
from pathlib import Path

import pandas as pd

input_path = Path("data/01_parish_list/parish_list.csv")
parishes_df = pd.read_csv(input_path)

print(f"Loaded {len(parishes_df)} parishes from {input_path}")
parishes_df.head()


Loaded 784 parishes from data\01_parish_list\parish_list.csv


,name,address,site_url,diocese,category,parish_list_url
0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",https://fatherpurcell.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
1,Holy Spirit Catholic Parish,"Montgomery, AL",https://holyspiritmgm.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",https://francisxseelos.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
3,Chapel of Our Lady of Bon Secour,"17266 County Road 49 South, Bon Secour, AL, 36511",https://www.stjohnms.com/our-lady-of-bon-secour,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
4,Christ the King Catholic Church,"711 College Avenue, Daphne, AL, 36526",https://ctkdaphne.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder


## Discover parish site pages
For the first three rows, check `/sitemap.xml` and sitemap declarations in `/robots.txt`, follow sitemap indexes, and also collect homepage/navigation links. Keep same-site HTTP(S) page URLs, remove fragments, exclude common file downloads, and deduplicate per parish. Query strings and distinct paths are preserved.

Downloads are cached in `data/02_extract_ministries/raw/`. Failures are recorded in `discovery_errors_df` so homepage discovery can still work when a sitemap is unavailable. This step discovers URLs; it does not download every discovered page.


In [7]:
from utils.page_discovery import discover_site_pages

PARISH_LIMIT = 10
output_dir = Path("data/02_extract_ministries")
cache_dir = output_dir / "raw"
selected_parishes = parishes_df.head(PARISH_LIMIT)
page_rows = []
error_rows = []

for parish_index, parish in selected_parishes.iterrows():
    site_url = parish["site_url"] if pd.notna(parish["site_url"]) else ""
    pages, errors = discover_site_pages(site_url, cache_dir)
    parish_info = {
        "parish_index": parish_index,
        "name": parish["name"],
        "diocese": parish["diocese"],
        "site_url": site_url,
    }
    page_rows.extend({**parish_info, **page} for page in pages)
    error_rows.extend({**parish_info, **error} for error in errors)
    print(f"{parish['name']}: {len(pages)} pages, {len(errors)} discovery issues")

parish_columns = ["parish_index", "name", "diocese", "site_url"]
site_pages_df = pd.DataFrame(page_rows, columns=parish_columns + ["page_url", "sources"])
discovery_errors_df = pd.DataFrame(error_rows, columns=parish_columns + ["url", "error"])
output_dir.mkdir(parents=True, exist_ok=True)
site_pages_df.to_csv(output_dir / "site_pages.csv", index=False)
discovery_errors_df.to_csv(output_dir / "discovery_errors.csv", index=False)
site_pages_df.head(20)


Father Purcell Memorial Center for Exceptional Children: 11 pages, 0 discovery issues
Holy Spirit Catholic Parish: 52 pages, 0 discovery issues
Blessed Francis Xavier Seelos Parish: 468 pages, 0 discovery issues
Chapel of Our Lady of Bon Secour: 263 pages, 1 discovery issues
Christ the King Catholic Church: 859 pages, 0 discovery issues
Our Lady of the Gulf Catholic Church: 232 pages, 0 discovery issues
Shrine of the Holy Cross & St. John Mission: 46 pages, 0 discovery issues
St Robert Bellarmine Catholic Church: 111 pages, 0 discovery issues
St. Agatha Parish: 38 pages, 0 discovery issues
St. Bartholomew: 29 pages, 0 discovery issues


,parish_index,name,diocese,site_url,page_url,sources
0,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/,"homepage, homepage_link, sitemap"
1,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/admission,"homepage_link, sitemap"
2,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/blog,sitemap
3,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/bulletins,sitemap
4,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/calendar,sitemap
5,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/contact,"homepage_link, sitemap"
6,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/events,sitemap
7,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/history,"homepage_link, sitemap"
8,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/news,sitemap
9,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/photoalbums,sitemap


In [6]:
# Inspect unavailable sitemaps or other requests that need attention.
discovery_errors_df


,parish_index,name,diocese,site_url,url,error
